In [1]:
import os

In [2]:
%pwd

'/home/lox_masade/Projects/MLOPS/dsp18/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/lox_masade/Projects/MLOPS/dsp18'

In [5]:
print(os.listdir())

['src', 'template.py', 'research', '.gitignore', 'main.py', 'LICENSE', 'README.md', 'templates', '.git', 'config', 'schema.yaml', '.github', 'params.yaml', 'Dockerfile', 'setup.py', 'dsp18env', 'requirements.txt', 'logs']


In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
from src.dsp18.constants import *
from src.dsp18.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(self, config_filepath=config_filepath, params_filepath=params_filepath, schema_filepath=schema_filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

In [9]:
import urllib.request as request
from src.dsp18 import logger
import zipfile

In [10]:
## Component-DataIngestion
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    # Downloading the zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"{self.config.local_data_file} already exists.")

    # Unzipping the downloaded file

    def unzip_data(self):
        """
        zip_file_path: str
        Extracts the contents of a zip file to a specified directory.
        Function returns None.
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Data unzipped to {self.config.unzip_dir}")

    

In [11]:
try:
    config=ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    print("Artifacts root exists:", Path("artifacts").exists())
    print("Data ingestion dir exists:", Path("artifacts/data_ingestion").exists())
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.unzip_data()

except Exception as e:
    # logger.error(f"Error occurred: {e}")
    raise e

[2025-08-29 01:58:24,616: INFO: common: YAML file config/config.yaml loaded successfully.]
[2025-08-29 01:58:24,624: INFO: common: YAML file params.yaml loaded successfully.]
[2025-08-29 01:58:24,628: INFO: common: YAML file schema.yaml loaded successfully.]
[2025-08-29 01:58:24,631: INFO: common: Created directory: artifacts]
[2025-08-29 01:58:24,635: INFO: common: Created directory: artifacts/data_ingestion]
Artifacts root exists: True
Data ingestion dir exists: True
[2025-08-29 01:58:25,015: INFO: 2529087064: artifacts/data_ingestion/data.zip downloaded! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: C5E6:9E030:32

In [12]:
p = Path("params.yaml")
print("exists:", p.exists(), "size:", p.stat().st_size if p.exists() else None)
print(p.read_text())

exists: True size: 154
# params.yaml
base:
  random_state: 42
  test_size: 0.2

data_ingestion:
  timeout_sec: 60
  retries: 3
  chunk_size: 1048576   # 1 MB
  verify_ssl: true



In [13]:
print(repr(Path("params.yaml").read_text()))

'# params.yaml\nbase:\n  random_state: 42\n  test_size: 0.2\n\ndata_ingestion:\n  timeout_sec: 60\n  retries: 3\n  chunk_size: 1048576   # 1 MB\n  verify_ssl: true\n'
